# QICKoDeS Tutorial — Running QICK Through QCoDeS

This notebook builds on **`test_1.ipynb`** (the raw-QICK tutorial) and explains
**qickodes** — `src/libraries/qickodes` in this repo — the [QCoDeS](https://github.com/microsoft/qcodes)
driver for QICK. It's written for two overlapping audiences: people who know some QICK but
have never used QCoDeS, and people who know QCoDeS but have never used QICK.

**You don't need to re-learn anything.** Every important piece of qickodes turns out to be
a thin, QCoDeS-flavored wrapper around a method from `test_1.ipynb` — `declare_gen`,
`declare_readout`, `add_gauss`, `trigger()`, `AcquireMixin.acquire()`. This notebook points
out each of those connections explicitly, with the actual source lines, as we go.

1. What QCoDeS is (with an analogy) and a beginner primer on its core concepts
2. What qickodes is, and how it connects to the board
3. DAC channels, ADC channels, envelopes, and pulses — each traced back to test_1.ipynb
4. Sequencing a shot with `macro_list`
5. Running a measurement, and what happens under the hood
6. Hardware sweeps vs. software sweeps, the QCoDeS way
7. Three full worked examples (frequency sweep, gain sweep, 2D sweep)
8. Reproducibility: `Station` and snapshots
9. Cheat sheet: every qickodes concept mapped to its raw-QICK equivalent
10. Glossary + where to go next

**Convention, same as `test_1.ipynb`:** cells marked **⚠️ requires hardware + qcodes/qickodes
installed** are written out fully and correctly but not executed here. Cells marked
**✅ runs anywhere** are plain Python and are actually run in this notebook.


## 1. What QCoDeS is

**Analogy:** think of QCoDeS as a **universal remote control with a lab notebook built in**,
for scientific instruments. In a real lab you might have a power supply, a magnet
controller, a lock-in amplifier, and (here) an RFSoC board -- all from different vendors,
all with different native APIs. QCoDeS wraps each one so that every physical setting
becomes a named "button" you press the *same way* no matter which instrument it belongs to:

```python
instrument.some_setting.set(value)   # press the button
instrument.some_setting.get()        # read the button
```

And every time you run a measurement, QCoDeS automatically writes down -- into a database,
alongside the results -- exactly which buttons were set to what. A logbook that writes
itself, so any measurement can be reproduced exactly, months later, without needing your
memory of what you did.

**qickodes is the specific remote-control faceplate built for the QICK board.** It doesn't
reimplement pulse generation, timing, or acquisition -- it takes every method you already
learned in `test_1.ipynb` (`declare_gen`, `set_pulse_registers`/`add_pulse`, `trigger()`,
`add_gauss`, `acquire()`) and relabels them as QCoDeS-standard objects, so the QICK board
can plug into the exact same measurement/sweep/database infrastructure as any other
instrument in a QCoDeS-based lab.


## 2. QCoDeS primer -- the four concepts you need

If you've never used QCoDeS, these four ideas cover essentially everything you'll see below.

### `Parameter` -- one named, labeled, unit-tagged, settable/gettable value

```python
some_instrument.frequency.set(1e9)   # or, shorthand: some_instrument.frequency(1e9)
f = some_instrument.frequency.get()  # or, shorthand: f = some_instrument.frequency()
```
A `Parameter` always has a `name`, a `label` (human-readable), and a `unit` -- and QCoDeS
validates every `.set()` against declared bounds automatically. The two shorthands above
(calling the parameter like a function) are used constantly in real qickodes code -- you'll
see `ge_pi_pulse.gain(0.52)` rather than `.gain.set(0.52)` in the worked examples below.

### `Instrument` / `InstrumentChannel` / `InstrumentModule` -- how Parameters get organized

**Analogy:** `Instrument` = a rack-mounted box. `InstrumentChannel`/`InstrumentModule` = one
physical output/channel on that box's front panel, or a named settings-group on that
channel. `Parameter` = one knob on that channel. This is a strict nesting, and qickodes
follows it exactly:

```
QickInstrument                  <- the whole board (an Instrument)
  .dacs[0]                      <- one DAC channel (an InstrumentChannel)
    .nqz                        <- a Parameter directly on the channel
    some_pulse                  <- a pulse defined on that channel (an InstrumentModule)
      .freq, .gain, .phase      <- Parameters on that pulse
```
A `ChannelTuple` (e.g. `qi.dacs`) is just an indexable/iterable collection of channels.

### `Station` -- the collection of instruments in your current experiment

```python
station = Station()
station.add_component(qi)   # qi = your QickInstrument
```
Mostly bookkeeping: it's what gets snapshotted (next point) alongside every measurement.

### `Measurement` + database + snapshot -- the self-writing logbook

```python
initialise_or_create_database_at(Path("my_experiment.db"))
experiment = load_or_create_experiment("experiment_name", "sample_name")
meas = Measurement(station=station, name="my_measurement")
```
When you run a `Measurement`, QCoDeS saves a **snapshot** -- the current value of *every*
`Parameter` on *every* instrument in the `Station` -- into the database, alongside the
result data. This is why qickodes represents every pulse setting as a real `Parameter`
rather than a plain Python variable: it's what makes "what exactly did I run three months
ago" a solved problem, automatically, for free.


In [ ]:
# ✅ runs anywhere -- a tiny mock demonstrating the param(value)/param() shorthand,
# the exact mechanism (a __call__ method) real qcodes.Parameter objects use, without
# needing qcodes installed.

class ToyParameter:
    def __init__(self, name, unit):
        self.name = name
        self.unit = unit
        self._value = None

    def set(self, value):
        print(f"  setting {self.name} = {value} {self.unit}")
        self._value = value

    def get(self):
        return self._value

    def __call__(self, *args):
        # this is the trick behind param(value) == param.set(value), param() == param.get()
        if args:
            return self.set(args[0])
        return self.get()


gain = ToyParameter("gain", "DAC unit")
gain(0.52)          # same as gain.set(0.52)
print(gain())        # same as gain.get()


## 3. What qickodes is, concretely

`src/libraries/qickodes` currently has **two parallel APIs**: an older one built on tProc
v1 (`instruments.py`, `channels.py`, ...) and a newer one built on tProc v2
(`instrument_v2.py`, `channels_v2.py`, ..., the `_v2`-suffixed files). The package's own
README says tProc v1 support is in maintenance-only mode and **tProc v2 is what you should
use** -- this notebook covers the v2 API exclusively.

**Import gotcha worth knowing:** the top-level `from qickodes import QickInstrument`
(`src/libraries/qickodes/src/qickodes/__init__.py`) currently still resolves to the *old*
v1 driver. The real example scripts (`example_scripts_v2/`) bypass this and import v2
directly:

```python
from qickodes.instrument_v2 import QickInstrument, SoftwareSweep
from qickodes.channels_v2 import DacChannel
from qickodes.pulse_base_v2 import DacPulse
from qickodes.pulses_v2 import *          # ConstantPulse, ArbitraryPulse, ...
from qickodes.envelope_base_v2 import DacEnvelope
from qickodes.envelopes_v2 import *       # GaussianEnvelope, GaussianDragEnvelope, ...
from qickodes.macros_v2 import *          # PlayPulse, Trigger, DelayAuto, ...
from qick.asm_v2 import QickSweep1D
```

### Connecting to the board

`QickInstrument.__init__` does exactly one QICK-level thing:
```python
self.soc, self.soccfg = make_proxy(ns_host, ns_port)
```
-- the *same* `make_proxy`/Pyro4 remote-control pattern from `test_1.ipynb` section 4. So
everything about "who runs where" from that section (compilation happens in your Python
process, only `soc.*` calls cross the network) applies unchanged here.

```python
from qickodes.instrument_v2 import QickInstrument
qi = QickInstrument("ip.address.of.board", ns_port=8888, name="qi")
print(qi.soccfg)   # exact same QickConfig object/printout as test_1.ipynb section 4
```


## 4. `QickInstrument`'s structure

On construction, `QickInstrument` builds:

- **`qi.dacs`** -- a `ChannelTuple` with one channel object per DAC (`StandardDacChannel`
  for ordinary channels, `MultiplexedDacChannel` for muxed generators -- picked
  automatically based on `soccfg['gens'][n]['type']`, the same generator-type field from
  `QickConfig.description()`).
- **`qi.adcs`** -- a `ChannelTuple` of `AdcChannel`, one per readout.
- **`qi.ddr4_buffer`** -- wraps the DDR4 streaming-capture hardware (a large continuous
  capture buffer, not covered in `test_1.ipynb` -- useful for long decimated captures that
  exceed the normal decimated buffer size from section 7.1).
- **`qi.hard_avgs`** / **`qi.soft_avgs`** -- direct QCoDeS `Parameter` equivalents of
  `test_1.ipynb`'s `reps` and `rounds`/`soft_avgs`: `hard_avgs` is how many repetitions the
  tProc does in hardware per program run; `soft_avgs` is how many times Python re-runs and
  averages the whole program (section 3's control-plane/data-plane distinction, and
  section 7.1's streaming-readout mechanism, are both unchanged underneath).
- **`qi.macro_list`** -- empty until you call `qi.set_macro_list([...])` (section 6 below);
  this is what replaces hand-writing `initialize()`/`body()`.


## 5. DAC and ADC channels -- traced back to `declare_gen`/`declare_readout`

### DAC channels

```python
qubit_dac = qi.dacs[1]
qubit_dac.nqz.set(1)                        # ManualParameter, same "Nyquist zone" concept
qubit_dac.digital_mixer_freq.set(5.5e9)     # only exists if this DAC has a mixer
```

Checking `channels_v2.py`, `StandardDacChannel.initialize()` does exactly this:
```python
program.declare_gen(self.channel_num, self.nqz.get(), mixer_freq)
```
**That's the literal `declare_gen` method from `test_1.ipynb` section 5/`AbsQickProgram`.**
Nothing new is implemented here -- `nqz`/`digital_mixer_freq` are just QCoDeS `Parameter`
front-ends for the exact arguments `declare_gen` already takes. One unit difference to
remember: qickodes Parameters are consistently in **SI base units** (Hz, sec, deg) -- the
`/1e6` conversions to MHz happen internally, right before the call into QICK.

### ADC channels

```python
readout_adc = qi.adcs[0]
readout_adc.freq.set(6e9)
readout_adc.length.set(10e-6)
readout_dac.matching_adc.set(readout_adc.channel_num)   # frequency-match, like gen_ch= in declare_readout
```

`AdcChannel.initialize()` calls:
```python
program.declare_readout(ch=self.channel_num, freq=..., phase=0, length=..., gen_ch=...)
```
**The same `declare_readout` from section 7.** `matching_dac`/`matching_adc` are exactly
the `gen_ch=`/`ro_ch=` frequency-matching arguments from sections 5 and 7 -- still solving
the same bit-exact frequency-matching problem `QickConfig.roundfreq`/`calc_fstep` handled.


## 6. Envelopes -- traced back to `add_gauss`

```python
from qickodes.envelopes_v2 import GaussianDragEnvelope

ge_envelope = GaussianDragEnvelope(qubit_dac, "ge_envelope")
ge_envelope.sigma.set(5e-9)     # seconds, not fabric clocks -- qickodes converts for you
ge_envelope.length.set(20e-9)
ge_envelope.delta.set(-161e6)   # anharmonicity, same parameter as add_DRAG's `delta`
ge_envelope.alpha.set(-0.12)
```

`GaussianEnvelope.initialize()` (the simpler, non-DRAG sibling) is almost embarrassingly
direct:
```python
program.add_gauss(ch=self.parent.channel_num, name=self.short_name,
                   sigma=self.sigma.get() * 1e6, length=self.length.get() * 1e6)
```
**The exact `add_gauss` from section 6.1**, called with the same `sigma`/`length`
parameters, just unit-converted from seconds to microseconds first. `GaussianDragEnvelope`
does the same thing via `add_DRAG`. Section 6.2's whole "envelope memory is loaded once,
before the tProc runs, and is a separate hardware block from the tProc" explanation applies
completely unchanged -- qickodes doesn't touch that architecture, it just gives you
`Parameter` objects instead of raw function arguments to build the same envelope data.


## 7. Pulses -- traced back to `set_pulse_registers`/`add_pulse`

Two pulse styles map onto the `"const"`/`"arb"` styles from section 6:

```python
from qickodes.pulses_v2 import ConstantPulse, ArbitraryPulse

readout_pulse = ConstantPulse(readout_dac, "readout_pulse")
readout_pulse.gain.set(1)          # -1..1, note: NOT a raw DAC code like test_1.ipynb's gain=20000
readout_pulse.freq.set(6e9)
readout_pulse.length.set(10e-6)

ge_pi_pulse = ArbitraryPulse(qubit_dac, "ge_pi_pulse", ge_envelope)   # needs an envelope
ge_pi_pulse.gain.set(0.520)
ge_pi_pulse.freq.set(3.43899e9)
```

`ConstantPulse.initialize()`:
```python
program.add_pulse(ch=..., name=..., style="const", freq=..., phase=..., gain=..., length=...,
                   phrst=..., stdysel=..., mode=...)
```
`add_pulse` is tProc v2's own sibling of v1's `set_pulse_registers` -- different method
name (different module: `qick.asm_v2` vs `asm_v1`), but the same parameter vocabulary you
already know: `style`, `freq`, `phase`, `gain`, `length`/`envelope`, `mode`
(`"oneshot"`/`"periodic"`, same as section 6's `mode=`), `stdysel`. `reset_phase` is
`phrst`. Gain here is normalized to **-1..1** rather than a raw DAC integer -- qickodes
always works in physical/normalized units and lets QICK do the register math, same
principle as `freq2reg` being invisible to you in `test_1.ipynb` once you call
`self.freq2reg(...)` yourself.

### `.copy()` -- a genuinely convenient qickodes-only addition

```python
ge_half_pi_pulse = ge_pi_pulse.copy("ge_half_pi_pulse")
ge_half_pi_pulse.gain.set(ge_pi_pulse.gain.get() / 2)
```
Clones every `Parameter` value into a new, independently-adjustable pulse object -- there's
no raw-QICK equivalent to this, it's a convenience `qickodes` adds on top.


## 8. Sequencing a shot: `macro_list` replaces hand-writing `body()`

In `test_1.ipynb`, you wrote `initialize()`/`body()` by hand, calling `pulse()`/`trigger()`/
`sync_all()` yourself in order. qickodes replaces that with a **declarative list**:

```python
from qickodes.macros_v2 import PlayPulse, Trigger, DelayAuto

qi.set_macro_list([
    DelayAuto(qi, 10e-9),
    Trigger(qi, readout_adc, t=450e-9),
    PlayPulse(qi, readout_pulse),
    DelayAuto(qi, 10e-9),
])
```

Each entry is a `Macro` object whose `create_qick_macro()` emits exactly the method you'd
expect from `test_1.ipynb`:

| Macro | Emits (tProc v2 equivalent of...) |
|---|---|
| `PlayPulse(qi, pulse, t="auto")` | `qick.asm_v2.Pulse(...)` -- the v2 sibling of `pulse()` (section 6) |
| `Trigger(qi, adcs, pins, t, width)` | `qick.asm_v2.Trigger(ros=..., pins=..., t=..., width=...)` -- **the same parameter names as `trigger()`** from section 7 |
| `DelayAuto(qi, t, wait_for_dacs, wait_for_adcs)` | `qick.asm_v2.Delay(..., auto=True, ...)` -- the v2 sibling of `sync_all()`/`wait_all()` |

Internally (`programs_v2.py`), `QickInstrument`'s own `AveragerProgram._body()` just does:
```python
for macro in self.qick_instrument.macro_list:
    self.append_macro(macro.create_qick_macro())
```
-- literally iterating your list and appending each macro's ASM, in order. It's the same
idea as writing `body()` by hand, just data-driven instead of imperative code, which is
what lets `macro_list` itself be part of the QCoDeS snapshot (section 10).


## 9. Running a measurement -- and what happens underneath

```python
qi.hard_avgs.set(1000)
qi.soft_avgs.set(1)

qi.run(
    Measurement(station=station, name="my_measurement"),
    acquisition_mode="accumulated",   # see options below
)
```

`acquisition_mode` options, and their `test_1.ipynb` equivalents:

| `acquisition_mode` | Equivalent from `test_1.ipynb` |
|---|---|
| `"accumulated"` | `acquire()` (section 5/7) -- averaged I/Q |
| `"decimated"` | `acquire_decimated()` (section 7) -- raw scope-like trace |
| `"accumulated shots"` | `get_shots()`-style single-shot data (mentioned in section 12's glossary) |
| `"ddr4"` | new: streams from the DDR4 buffer (section 4), for long captures |
| `"accumulated geometric median"` / `"state population"` | qickodes-only additions: robust averaging and qubit-state histograms via a classifier function -- no raw-QICK equivalent in `test_1.ipynb` |

**The important part**, straight from `instrument_v2.py`:
```python
program = AveragerProgram(self, hardware_loop_counts)   # qickodes' own subclass
all_iq = qick.qick_asm.AcquireMixin.acquire(self=program, soc=self.soc,
                                             rounds=self.soft_avgs.get(), progress=progress)
```
**This is the literal `AcquireMixin.acquire()` method from `test_1.ipynb` section 7.1** --
same circular accumulated buffer, same background streaming thread polling
`get_tproc_counter()`/draining with `get_accumulated()`, same overflow guard. `qi.run()`
does not reimplement any of that pipeline; it only handles registering QCoDeS `Parameter`s
as dataset columns and calling `datasaver.add_result(...)` with what `acquire()`/
`acquire_decimated()` already handed back.


## 10. Hardware sweeps vs. software sweeps -- the same decision rule as section 6.2/8.1

Section 6.2 of `test_1.ipynb` established a decision rule: does the swept quantity plug
into a register (`freq`/`gain`/`phase`/`length`) -- hardware-sweep it, fast; or does it
change the envelope *shape itself* -- Python-loop it, slower. qickodes exposes exactly this
choice, just through QCoDeS objects instead of `QickSweep`/`add_sweep` calls.

### Hardware sweep -- `QickSweep1D` + `hardware_loop_counts`

```python
from qick.asm_v2 import QickSweep1D

ge_pi_pulse.gain(QickSweep1D("gain", 0.50, 0.55))   # note: gain(...) shorthand for .set(...)

qi.run(meas, hardware_loop_counts={"gain": 101})    # 101 points; the loop's NAME ties them together
```
`QickSweep1D("gain", start, stop)` gives a *named* loop axis (matching the key you later
use in `hardware_loop_counts`); the point *count* is supplied separately, in
`hardware_loop_counts`. This is functionally the same thing as `test_1.ipynb`'s
`QickSweep(prog, reg, start, stop, expts)` + `add_sweep()` from sections 8/8.1 -- register
value stepped by the tProc itself, nanoseconds between points, no Python involved mid-run
-- just split across two places instead of one call.

Setting a `SweepableParameter` to a `QickSweep1D` registers it, under the hood, into
`qi.swept_params` (checking `parameters_v2.py`'s `SweepableParameter.set_raw`) -- and
`.get()` on a swept parameter returns the *rounded actual* value after hardware
quantization, exactly the `freq2reg`/`roundfreq` rounding theme from `test_1.ipynb` sections
1 and 3, just automatic here.

### Software sweep -- `SoftwareSweep`

```python
from qickodes.instrument_v2 import SoftwareSweep

qi.run(
    meas,
    software_sweeps=[SoftwareSweep([readout_pulse.freq, readout_adc.freq], 5.5e9, 6.5e9, 101)],
)
```
This is the QCoDeS-flavored version of `test_1.ipynb` section 6.2's `GaussPoint` Python
`for` loop: `qi.run()` iterates the values itself, calling `.set()` on each listed
parameter and re-running the whole program (fresh `config_all()`, fresh envelope reload if
relevant) for every point -- milliseconds between points, not nanoseconds. Passing *two*
parameters (`readout_pulse.freq` and `readout_adc.freq`) sweeps them together in lockstep,
the software-sweep equivalent of section 8.1's `merge_sweeps`.

**Same decision rule as before:** register-only quantity, one hardware axis -> `QickSweep1D`
+ `hardware_loop_counts`. Multiple hardware axes -> multiple `hardware_loop_counts` entries
(worked example 3, below) -- the qickodes equivalent of `NDAveragerProgram`/`add_sweep`
ordering from section 8.1, except here **there is no ordering step to get right** --
`hardware_loop_counts` is a dict, and `AveragerProgram._initialize()` just calls
`self.add_loop(name, count)` for each entry; the nesting is handled for you. Envelope-shape
change, or anything that can't be a register -> `SoftwareSweep`.


## 11. Worked example 1: frequency sweep (software sweep)

Directly adapted from the real `example_scripts_v2/meas_s21.py` in this repo.


In [ ]:
# ⚠️ requires hardware + qcodes/qickodes installed -- S21 vs frequency.
from pathlib import Path
from qcodes import Measurement, Station, initialise_or_create_database_at, load_or_create_experiment
from qickodes.instrument_v2 import QickInstrument, SoftwareSweep
from qickodes.pulses_v2 import ConstantPulse
from qickodes.macros_v2 import Trigger, PlayPulse

initialise_or_create_database_at(Path("qickodes_tutorial.db"))
experiment = load_or_create_experiment("qickodes_tutorial", "example_sample")

qi = QickInstrument("ip.address.of.board", name="qi")
station = Station()
station.add_component(qi)

readout_dac = qi.dacs[0]
readout_adc = qi.adcs[0]
readout_dac.matching_adc.set(readout_adc.channel_num)
readout_adc.matching_dac.set(readout_dac.channel_num)
readout_dac.nqz.set(1)

readout_pulse = ConstantPulse(readout_dac, "readout_pulse")
readout_pulse.gain.set(1)
readout_pulse.freq.set(6e9)
readout_pulse.length.set(100e-6)
readout_adc.freq.set(readout_pulse.freq.get())
readout_adc.length.set(readout_pulse.length.get())

qi.set_macro_list([
    Trigger(qi, readout_adc, t=0),
    PlayPulse(qi, readout_pulse),
])

qi.hard_avgs.set(1000)
qi.soft_avgs.set(1)

run_id = qi.run(
    Measurement(station=station, name="meas_s21"),
    software_sweeps=[SoftwareSweep([readout_pulse.freq, readout_adc.freq], 5.5e9, 6.5e9, 101)],
)
print(f"run_id = {run_id}")


## 12. Shared setup for examples 2 and 3 (a qubit drive line)

The real `example_scripts_v2/` scripts all share one `header.py` that sets up the wiring
once, then each measurement script just imports it (`from header import *`). Examples 2
and 3 below both need a qubit drive pulse and a `readout()` helper, so -- mirroring that
same pattern -- here is that shared setup, once, continuing from example 1's `qi`/`station`:


In [ ]:
# ⚠️ requires hardware + qcodes/qickodes installed -- shared qubit-drive setup,
# continuing from example 1's qi/station. Mirrors example_scripts_v2/header.py.
from qickodes.envelopes_v2 import GaussianDragEnvelope
from qickodes.pulses_v2 import ArbitraryPulse
from qickodes.macros_v2 import DelayAuto, Trigger, PlayPulse

def readout():
    return [
        DelayAuto(qi, 10e-9),
        Trigger(qi, readout_adc, t=450e-9),
        PlayPulse(qi, readout_pulse),
        DelayAuto(qi, 10e-9),
    ]

qubit_dac = qi.dacs[1]
qubit_dac.nqz.set(1)

ge_envelope = GaussianDragEnvelope(qubit_dac, "ge_envelope")
ge_envelope.sigma.set(5e-9)
ge_envelope.length.set(20e-9)
ge_envelope.delta.set(-161e6)
ge_envelope.alpha.set(-0.12)

ge_pi_pulse = ArbitraryPulse(qubit_dac, "ge_pi_pulse", ge_envelope)
ge_pi_pulse.gain.set(0.520)
ge_pi_pulse.freq.set(3.43899e9)


## 13. Worked example 2: pi-pulse gain sweep (hardware sweep)

Directly adapted from `example_scripts_v2/meas_ge_pi_pulse_gain_sweep.py`. The idea: play
many pi-pulses in a row before reading out, so a small gain error accumulates into a
visible drift, letting you pick out the exact optimal gain -- a real calibration trick, not
a qickodes-specific one.


In [ ]:
# ⚠️ requires hardware + qcodes/qickodes installed -- pi-pulse gain sweep, hardware-swept.
# Continues from the shared setup above (qi, ge_pi_pulse, readout()).
from qick.asm_v2 import QickSweep1D
from qickodes.macros_v2 import PlayPulse

ge_pi_pulse.gain(QickSweep1D("gain", 0.50, 0.55))   # register-only -> fully hardware-swept

qi.set_macro_list([
    *([PlayPulse(qi, ge_pi_pulse)] * 20),  # repeat 20x to amplify gain error
    *readout(),
])

qi.hard_avgs.set(1000)
qi.final_delay.set(200e-6)

run_id = qi.run(
    Measurement(station=station, name="meas_ge_pi_pulse_gain_sweep"),
    hardware_loop_counts={"gain": 101},
)
print(f"run_id = {run_id}")


## 14. Worked example 3: 2D hardware sweep (frequency x gain)

The qickodes equivalent of `test_1.ipynb` section 8.1's `NDAveragerProgram` frequency x
gain map -- two independent `hardware_loop_counts` entries, no manual `add_sweep`
ordering needed.


In [ ]:
# ⚠️ requires hardware + qcodes/qickodes installed -- 2D map: frequency x gain.
# Continues from the shared setup two cells above (qi, qubit_dac, readout()).
from qick.asm_v2 import QickSweep1D
from qickodes.pulses_v2 import ConstantPulse
from qickodes.macros_v2 import PlayPulse

probe_pulse = ConstantPulse(qubit_dac, "probe_pulse")
probe_pulse.length.set(2e-6)
probe_pulse.freq(QickSweep1D("freq", 3.40e9, 3.48e9))
probe_pulse.gain(QickSweep1D("gain", 0.05, 0.60))

qi.set_macro_list([
    PlayPulse(qi, probe_pulse),
    *readout(),
])

qi.hard_avgs.set(500)

run_id = qi.run(
    Measurement(station=station, name="meas_pulse_probe_2d"),
    hardware_loop_counts={"freq": 41, "gain": 15},   # order in the dict does not matter
)
print(f"run_id = {run_id}")


## 15. Reproducibility: `Station` and snapshots

This is the payoff for representing every setting as a `Parameter` (section 2) rather than
a plain variable. Right before running, `qi.run()` calls `self.snapshot(update=True)` --
this walks every `Instrument`/`InstrumentChannel`/`InstrumentModule` in the `Station` and
records the current value of *every* `Parameter`: every DAC's `nqz`, every pulse's
`freq`/`gain`/`phase`/`length`, every envelope's `sigma`, `hard_avgs`, `soft_avgs`, all of
it -- into the dataset's metadata, automatically, no extra code from you.

This is exactly why `qickodes`'s README states its design goal as "pulses and pulse
sequences should be parameterized such that they can be fully reproduced based on the
QCoDeS snapshot." A raw-QICK program (`test_1.ipynb`) is just Python code you wrote --
reproducing it means finding and re-reading that code. A qickodes measurement's *complete*
configuration is saved as data, alongside the results, every single run.


## 16. Cheat sheet: qickodes concept -> raw-QICK equivalent (`test_1.ipynb`)

| qickodes | test_1.ipynb equivalent |
|---|---|
| `QickInstrument(ns_host, ns_port)` | `make_proxy(...)`, section 4 |
| `qi.soccfg` | `soccfg` (`QickConfig`), sections 1-3 |
| `DacChannel`/`AdcChannel` `.initialize()` | `declare_gen`/`declare_readout`, sections 5, 7 |
| `qi.hard_avgs` | `reps` |
| `qi.soft_avgs` | `soft_avgs`/`rounds` |
| `ConstantPulse`/`ArbitraryPulse` `.initialize()` | `set_pulse_registers` (v1) / `add_pulse` (v2), section 6 |
| `GaussianEnvelope`/`GaussianDragEnvelope` `.initialize()` | `add_gauss`/`add_DRAG`, section 6.1 |
| `PlayPulse` macro | `pulse()`, section 6 |
| `Trigger` macro | `trigger()`, section 7 |
| `DelayAuto` macro | `sync_all()`/`wait_all()`, section 6 |
| `macro_list` | hand-written `body()`, section 5 |
| `qi.run(acquisition_mode="accumulated")` | `AcquireMixin.acquire()`, section 7.1 |
| `qi.run(acquisition_mode="decimated")` | `AcquireMixin.acquire_decimated()`, section 7 |
| `QickSweep1D` + `hardware_loop_counts` | `QickSweep` + `add_sweep`, sections 8, 8.1 |
| `SoftwareSweep` | the Python `for` loop pattern, section 6.2 |
| envelope memory, accumulated buffer, background streaming thread | unchanged -- section 7.1's whole explanation still applies underneath |


## 17. Glossary of QCoDeS-specific terms (not covered in `test_1.ipynb`)

| Term | Meaning |
|---|---|
| **Parameter** | a named, labeled, unit-tagged, validated, settable/gettable value |
| **Instrument** | the top-level object wrapping one piece of hardware (here, the whole QICK board) |
| **InstrumentChannel / InstrumentModule** | a sub-object grouping related Parameters (a DAC channel, a pulse, an envelope) |
| **ChannelTuple** | an indexable/iterable collection of channels (`qi.dacs`, `qi.adcs`) |
| **Station** | the set of instruments present in the current experiment |
| **Measurement** | a QCoDeS object representing one dataset-producing run |
| **Snapshot** | an automatic, complete record of every Parameter's value, saved with each dataset |
| **Database / experiment** | where QCoDeS datasets are persisted (a local SQLite file, by default) |
| **`param(value)` / `param()`** | shorthand for `param.set(value)` / `param.get()` |


## 18. Where to go next

- **In this repo**: `src/libraries/qickodes/example_scripts_v2/` has many more complete,
  runnable examples -- `meas_t1.py`, `meas_t2ramsey.py`, `meas_t2echo.py`,
  `meas_punchout.py`, `meas_single_shot_readout_vs_qubit_state.py`,
  `meas_s21_decimated.py`, `meas_s21_ddr4.py` -- all following the same `header.py` +
  `set_macro_list` + `run()` pattern covered here.
- **`src/libraries/qickodes/README.md`** -- installation, and the case for tProc v2 over v1.
- **QCoDeS documentation** (`https://microsoft.github.io/Qcodes/`) -- for anything
  QCoDeS-general not specific to qickodes: plotting datasets, `qcodes.dataset` querying,
  writing drivers for other lab instruments.
- **`test_1.ipynb`** -- everything in this notebook assumed you'd already read it; if any
  of the "traced back to..." callbacks above weren't clear, that's the place to revisit.
